In [1]:
import pandas as pd

df = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})
df

,a,b
0,1,4
1,2,5
2,3,6


In [2]:
df["c"]  # column "c" doesn't exist

KeyError: 'c'

In [10]:
import pandas as pd

df = pd.read_csv("../data/transactions.csv")

df["date"] = pd.to_datetime(df["date"], format="%m/%d/%Y %H:%M", errors="coerce")

s = df["amount"].str.strip()
is_neg = s.str.startswith("(")
vals = s.str.replace(r"[$,()]", "", regex=True).str.strip().astype(float)
df["amount"] = vals.mask(is_neg, -vals)

assert len(df) > 0, "transactions file is empty"
assert {"date", "amount", "vendor"}.issubset(df.columns), \
    f"missing required columns; got {df.columns.tolist()}"
assert df["amount"].notna().all(), "amount column has nulls"
assert (df["amount"] >= 0).all(), "negative amounts not allowed"

AssertionError: negative amounts not allowed

In [22]:
import pandas as pd
import pandera.pandas as pa
from pandera.typing import Series

class TransactionSchema(pa.DataFrameModel):
    date: Series[pa.DateTime] = pa.Field(nullable=True)
    # negatives are refunds/credits — no lower bound
    amount: Series[float]
    vendor: Series[str] = pa.Field(str_length={"min_value": 1})

df = pd.read_csv("../data/transactions.csv")

df["date"] = pd.to_datetime(df["date"], format="%m/%d/%Y %H:%M", errors="coerce")

s = df["amount"].str.strip()
is_neg = s.str.startswith("(")
vals = s.str.replace(r"[$,()]", "", regex=True).str.strip().astype(float)
df["amount"] = vals.mask(is_neg, -vals)

TransactionSchema.validate(df, lazy=True)

,date,amount,vendor
0,2014-07-19,450.00,10043564
1,2014-11-13,204.80,#18 VS
2,NaT,364.07,#22 PREFERRED PUMP & EQUI
3,2014-07-29,33.00,#44 BRAVO| MEMORIAL SQUA
4,2014-08-14,41.88,#44 BRAVO| MEMORIAL SQUA
...,...,...,...
437574,NaT,348.15,ZVETCO
437575,2014-08-13,515.00,ZYAGEN
437576,NaT,1711.00,ZYAGEN
437577,NaT,1100.00,ZYAGEN


In [5]:
import pandas as pd
import pandera.pandas as pa
from pandera.typing import Series

class TransactionSchema(pa.DataFrameModel):
    date: Series[pa.DateTime]
    amount: Series[float] = pa.Field(ge=0)
    vendor: Series[str] = pa.Field(str_length={"min_value": 1})

df = pd.read_csv("../data/transactions.csv")

df["date"] = pd.to_datetime(df["date"], format="%m/%d/%Y %H:%M", errors="coerce")

s = df["amount"].str.strip()
is_neg = s.str.startswith("(")
vals = s.str.replace(r"[$,()]", "", regex=True).str.strip().astype(float)
df["amount"] = vals.mask(is_neg, -vals)

TransactionSchema.validate(df, lazy=True)

SchemaErrors: {
    "SCHEMA": {
        "SERIES_CONTAINS_NULLS": [
            {
                "schema": "TransactionSchema",
                "column": "date",
                "check": "not_nullable",
                "error": "non-nullable series 'date' contains null values:2        NaT6        NaT7        NaT8        NaT9        NaT          ..437571   NaT437574   NaT437576   NaT437577   NaT437578   NaTName: date, Length: 222881, dtype: datetime64[us]"
            }
        ]
    },
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": "TransactionSchema",
                "column": "amount",
                "check": "greater_than_or_equal_to(0)",
                "error": "Column 'amount' failed element-wise validator number 0: greater_than_or_equal_to(0) failure cases: -150.0, -139.0, -500.0, -40.0, -21.68, -200.95, -39.0, -100.0, -103.0, -158.0, -24.0, -6.68, -2767.5, -901.05, -229.82, -1182.99, -732.0, -94.0, -23.61, -26.08, -10.34, -5.17, -2.75, -11.0

In [20]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)

np.random.rand(3)

array([0.37454012, 0.95071431, 0.73199394])

In [21]:
df = df.sort_values(["date", "vendor"]).reset_index(drop=True)

In [23]:
df.sample(10)

,date,amount,vendor
385740,NaT,214.20,UNITED 0167511148061
157401,NaT,-2.99,FEDEX 773384119599
362641,NaT,49.91,SW RURAL ELEC ASSOC
421567,NaT,264.00,WRIGHT COMFORT SOL
343358,NaT,22.79,STAPLES
275593,NaT,1437.46,OKLA RESORTS PARKS & GOL
253416,2014-08-12,-38.57,MSC
350957,NaT,38.98,STAPLES DIRECT
373125,2014-09-09,63.96,THE HOME DEPOT 3906
73827,2014-12-11,30.00,AT&T DATA


In [1]:
import logging
logging.basicConfig(level=logging.INFO, filename="run.log")

logging.info("pulled %d filings", n)
logging.error("skipped %s — parse failed", cik)

NameError: name 'n' is not defined